# v3 SFT — Intent + Retrieval Training

Continues from v2 LoRA adapter with the merged v2 dataset (~1,000 examples).
Adds `intent` (overview|specific|chat|followup) and `retrieval` (summary_only|with_chunks) fields.

- Lower LR (5e-5) to avoid catastrophic forgetting
- 3 epochs on the full merged dataset
- Loads from v2 LoRA checkpoint

## 1. Load Existing LoRA

In [1]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# Load the LoRA adapter from v3 training run
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./outputs/s1_sft_v3/final_lora",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

free, total = torch.cuda.mem_get_info(0)
print(f"VRAM: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")
print(f"Loaded LoRA from ./outputs/s1_sft_v3/final_lora")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0403 21:53:20.918000 13236 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 760/760 [00:09<00:00, 81.09it/s] 


VRAM: 12.0 GB used / 17.1 GB total
Loaded LoRA from ./outputs/s1_sft_v3/final_lora


## 2. Load v2 Merged Dataset

In [4]:
from datasets import load_dataset
from pathlib import Path

TD = Path(r"D:\Development\acervo-graph-model\training_data\v3")

# Load merged v3 dataset (v2 full + v3 new examples)
train_dataset = load_dataset(
    "json", data_files=str(TD / "s1_v3_full_training.jsonl"), split="train"
)
print(f"Training: {len(train_dataset)} examples")

val_dataset = load_dataset(
    "json", data_files=str(TD / "s1_v3_full_validation.jsonl"), split="train"
)
print(f"Validation: {len(val_dataset)} examples")

# Quick schema check
import json
sample = json.loads(train_dataset[0]["messages"][2]["content"])
assert "intent" in sample, "Missing intent field — did you run the migration?"
assert "retrieval" in sample, "Missing retrieval field — did you run the migration?"
print(f"\nSample output fields: {list(sample.keys())}")
print(f"Sample intent: {sample['intent']}, retrieval: {sample['retrieval']}")

Generating train split: 1058 examples [00:00, 44253.16 examples/s]


Training: 1058 examples


Generating train split: 101 examples [00:00, 11224.52 examples/s]

Validation: 101 examples

Sample output fields: ['intent', 'topic', 'retrieval', 'entities', 'relations', 'facts']
Sample intent: overview, retrieval: summary_only


## 3. Format for Training

In [5]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_formatted = train_dataset.map(format_example)
val_formatted = val_dataset.map(format_example)

print(f"Formatted {len(train_formatted)} train, {len(val_formatted)} val")

# Check token lengths
inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
lengths = [len(inner_tokenizer.encode(ex["text"])) for ex in train_formatted]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
over_limit = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
if over_limit:
    print(f"WARNING: {over_limit} examples exceed {MAX_SEQ_LENGTH} tokens — consider increasing MAX_SEQ_LENGTH")
else:
    print(f"All examples fit within {MAX_SEQ_LENGTH} tokens")

Map: 100%|██████████| 101/101 [00:00<00:00, 2168.68 examples/s]


Formatted 1058 train, 101 val
Token lengths: min=300, max=1268, avg=610
All examples fit within 2048 tokens


## 4. Train

In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        output_dir="./outputs/s1_sft_v3",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,           # 3 epochs — many new examples to learn
        learning_rate=5e-5,           # same as v2 — low LR for refinement
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        dataset_num_proc=None,
        dataloader_num_workers=0,
        report_to="none",
    ),
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Training {len(train_formatted)} examples x 3 epochs")
print(f"Estimated steps: {len(train_formatted) * 3 // (2 * 4)}")

Unsloth: Tokenizing ["text"]: 100%|██████████| 101/101 [00:00<00:00, 638.27 examples/s]


Trainable params: 29,097,984
Training 1058 examples x 3 epochs
Estimated steps: 396


In [7]:
stats = trainer.train()
print(f"\nTraining complete.")
print(f"  Total steps: {stats.global_step}")
print(f"  Train loss:  {stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,058 | Num Epochs = 3 | Total steps = 399
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,097,984 of 9,438,911,728 (0.31% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.072194,0.087039
100,0.067624,0.085218
150,0.056073,0.087302
200,0.052765,0.086497
250,0.055976,0.084524
300,0.044740,0.089758
350,0.043175,0.088923



Training complete.
  Total steps: 399
  Train loss:  0.0568


## 5. Quick Test — v2 Schema (intent + retrieval)

In [ ]:
import json
import sys
sys.path.insert(0, str(Path.cwd().parent / "01_dataset"))
from schema import S1_SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

# Test cases covering all 4 intents
test_cases = [
    {
        "label": "overview",
        "messages": [
            {"role": "system", "content": S1_SYSTEM_PROMPT},
            {"role": "user", "content": "EXISTING NODES:\n[{\"id\": \"beacon\", \"label\": \"Beacon\", \"type\": \"project\", \"layer\": \"PERSONAL\"}]\n\nTOPIC HINT: unresolved — classify the topic yourself\nCURRENT TOPIC: null\n\nPREVIOUS ASSISTANT: null\nUSER: What is this project about?"},
        ],
    },
    {
        "label": "specific",
        "messages": [
            {"role": "system", "content": S1_SYSTEM_PROMPT},
            {"role": "user", "content": "EXISTING NODES:\n[{\"id\": \"beacon\", \"label\": \"Beacon\", \"type\": \"project\", \"layer\": \"PERSONAL\"}, {\"id\": \"react\", \"label\": \"React\", \"type\": \"technology\", \"layer\": \"UNIVERSAL\"}]\n\nTOPIC HINT: same (high confidence from keyword match)\nCURRENT TOPIC: Beacon development\n\nPREVIOUS ASSISTANT: Beacon is built with React.\nUSER: How does the auth module work?"},
        ],
    },
    {
        "label": "chat",
        "messages": [
            {"role": "system", "content": S1_SYSTEM_PROMPT},
            {"role": "user", "content": "EXISTING NODES:\n[{\"id\": \"beacon\", \"label\": \"Beacon\", \"type\": \"project\", \"layer\": \"PERSONAL\"}]\n\nTOPIC HINT: same (high confidence from keyword match)\nCURRENT TOPIC: Beacon development\n\nPREVIOUS ASSISTANT: Here is an overview of the project.\nUSER: That's interesting, thanks!"},
        ],
    },
    {
        "label": "followup",
        "messages": [
            {"role": "system", "content": S1_SYSTEM_PROMPT},
            {"role": "user", "content": "EXISTING NODES:\n[{\"id\": \"beacon\", \"label\": \"Beacon\", \"type\": \"project\", \"layer\": \"PERSONAL\"}, {\"id\": \"react\", \"label\": \"React\", \"type\": \"technology\", \"layer\": \"UNIVERSAL\"}]\n\nTOPIC HINT: same (high confidence from keyword match)\nCURRENT TOPIC: Beacon development\n\nPREVIOUS ASSISTANT: The auth module uses JWT tokens with 24-hour expiry.\nUSER: Tell me more about that."},
        ],
    },
]

inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

for tc in test_cases:
    text = inner_tokenizer.apply_chat_template(
        tc["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"\n--- Expected: {tc['label']} ---")
    try:
        parsed = json.loads(response)
        got_intent = parsed.get('intent', 'MISSING')
        got_retrieval = parsed.get('retrieval', 'MISSING')
        mark = 'PASS' if got_intent == tc['label'] else 'FAIL'
        print(f"  [{mark}] intent={got_intent}, retrieval={got_retrieval}")
        print(f"  entities={len(parsed.get('entities', []))}, facts={len(parsed.get('facts', []))}")
    except json.JSONDecodeError as e:
        print(f"  [FAIL] JSON parse error: {e}")
        print(f"  Raw: {response[:200]}")


--- Expected: overview ---
  [PASS] intent=overview, retrieval=summary_only
  entities=0, facts=0

--- Expected: specific ---
  [FAIL] intent=followup, retrieval=with_chunks
  entities=0, facts=0

--- Expected: chat ---
  [PASS] intent=chat, retrieval=summary_only
  entities=0, facts=0

--- Expected: followup ---
  [PASS] intent=followup, retrieval=with_chunks
  entities=0, facts=0


## 6. Benchmark Failure Test (9 cases from v0.4)

In [9]:
import json
from pathlib import Path

failures_path = Path.cwd().parent / "01_dataset" / "benchmark_failures.jsonl"
if not failures_path.exists():
    print("benchmark_failures.jsonl not found")
else:
    with open(failures_path, encoding="utf-8") as f:
        failure_cases = [json.loads(line) for line in f if line.strip()]

    print(f"Testing {len(failure_cases)} v0.4 failure cases...\n")
    inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    FastLanguageModel.for_inference(model)

    passed = 0
    for i, case in enumerate(failure_cases):
        messages = case["messages"][:2]
        expected = json.loads(case["messages"][2]["content"])
        expected_intent = expected["intent"]

        text = inner_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

        user_msg = messages[1]["content"].split("USER: ")[-1][:60]

        try:
            parsed = json.loads(response)
            got_intent = parsed.get("intent", "MISSING")
            mark = "PASS" if got_intent == expected_intent else "FAIL"
            if got_intent == expected_intent:
                passed += 1
            print(f"  [{mark}] '{user_msg}' -> expected={expected_intent}, got={got_intent}")
        except json.JSONDecodeError:
            print(f"  [FAIL] '{user_msg}' -> JSON parse error")

    print(f"\nResult: {passed}/{len(failure_cases)} passed")
    if passed == len(failure_cases):
        print("All v0.4 failures are FIXED!")
    else:
        print(f"{len(failure_cases) - passed} failures remain — need more training data for these patterns")

Testing 9 v0.4 failure cases...

  [PASS] 'What is this application about?' -> expected=overview, got=overview
  [PASS] 'That's interesting' -> expected=chat, got=chat
  [PASS] 'How are tests organized?' -> expected=overview, got=overview
  [PASS] 'What is this book about?' -> expected=overview, got=overview
  [PASS] 'These are great stories' -> expected=chat, got=chat
  [PASS] 'What genre would you classify these?' -> expected=overview, got=overview
  [PASS] 'What project is documented here?' -> expected=overview, got=overview
  [PASS] 'What project is this about?' -> expected=overview, got=overview
  [PASS] 'Thanks for the overview' -> expected=chat, got=chat

Result: 9/9 passed
All v0.4 failures are FIXED!


## 7. Save Model

In [10]:
model.save_pretrained("./outputs/s1_sft_v3/final_lora")
tokenizer.save_pretrained("./outputs/s1_sft_v3/final_lora")
print("LoRA v3 saved to ./outputs/s1_sft_v3/final_lora")

LoRA v3 saved to ./outputs/s1_sft_v3/final_lora


In [11]:
# ── 8a. Merge LoRA + Export GGUF (local) ──
import subprocess
from pathlib import Path

# Step 0: Merge LoRA into full model (needed for GGUF conversion)
print("--- Merging LoRA into full model ---")
model.save_pretrained_merged(
    "./outputs/s1_sft_v3/merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved.\n")

src = Path("./outputs/s1_sft_v3/merged")
out = Path("./outputs/s1_sft_v3/gguf")
out.mkdir(exist_ok=True)

# Find llama.cpp tools
llama_dir = Path.home() / ".unsloth" / "llama.cpp"
quantize_bin = None
convert_script = None

for name in ["llama-quantize.exe", "quantize.exe"]:
    for d in [llama_dir, llama_dir / "build" / "bin" / "Release"]:
        p = d / name
        if p.exists():
            quantize_bin = p
            break
    if quantize_bin:
        break

for name in ["convert-hf-to-gguf.py", "convert_hf_to_gguf.py"]:
    p = llama_dir / name
    if p.exists():
        convert_script = p
        break

print(f"quantize: {quantize_bin}")
print(f"convert:  {convert_script}")
assert quantize_bin and quantize_bin.exists(), f"llama-quantize not found"
assert convert_script and convert_script.exists(), f"convert script not found"

# Step 1: Convert to BF16 GGUF
print("\n--- Converting to BF16 GGUF ---")
bf16_path = out / "acervo-extractor-v3-bf16.gguf"
result = subprocess.run(
    ["python", str(convert_script), str(src),
     "--outfile", str(bf16_path), "--outtype", "bf16"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=600
)
if result.stdout:
    print(result.stdout[-300:])
if result.returncode != 0:
    print("ERROR:", result.stderr[-500:])
    raise RuntimeError("BF16 conversion failed")
print(f"BF16 GGUF: {bf16_path} ({bf16_path.stat().st_size / 1e9:.1f} GB)")

# Step 2: Quantize to Q4_K_M
print("\n--- Quantizing to Q4_K_M ---")
q4_path = out / "acervo-extractor-v3-Q4_K_M.gguf"
result = subprocess.run(
    [str(quantize_bin), str(bf16_path), str(q4_path), "Q4_K_M"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=600
)
if result.stdout:
    print(result.stdout[-300:])
if result.returncode != 0:
    print("ERROR:", result.stderr[-500:])
    raise RuntimeError("Quantization failed")

print(f"\nDone! Local GGUF ready:")
print(f"  BF16: {bf16_path} ({bf16_path.stat().st_size / 1e9:.1f} GB)")
print(f"  Q4_K_M: {q4_path} ({q4_path.stat().st_size / 1e9:.1f} GB)")
print(f"\nTest locally with: ollama create acervo-v3 -f Modelfile")

--- Merging LoRA into full model ---
Found HuggingFace hub cache directory: C:\Users\sandy\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  3.05it/s]


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `./outputs/s1_sft_v3/merged`: 100%|██████████| 4/4 [00:10<00:00,  2.63s/it]


Successfully copied all 4 files from cache to `./outputs/s1_sft_v3/merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:35<00:00,  8.86s/it]


Unsloth: Merge process complete. Saved to `d:\Development\acervo-graph-model\02_training\outputs\s1_sft_v3\merged`
Merged model saved.

quantize: C:\Users\sandy\.unsloth\llama.cpp\build\bin\Release\llama-quantize.exe
convert:  C:\Users\sandy\.unsloth\llama.cpp\convert_hf_to_gguf.py

--- Converting to BF16 GGUF ---
BF16 GGUF: outputs\s1_sft_v3\gguf\acervo-extractor-v3-bf16.gguf (17.9 GB)

--- Quantizing to Q4_K_M ---

main: quantize time = 140730.73 ms
main:    total time = 140730.73 ms


Done! Local GGUF ready:
  BF16: outputs\s1_sft_v3\gguf\acervo-extractor-v3-bf16.gguf (17.9 GB)
  Q4_K_M: outputs\s1_sft_v3\gguf\acervo-extractor-v3-Q4_K_M.gguf (5.6 GB)

Test locally with: ollama create acervo-v3 -f Modelfile


## 8b. Publish to HuggingFace (optional, run only after local testing)

In [ ]:
# ── 8b. Publish to HuggingFace ──
# Only run this AFTER you've tested the GGUF locally and are happy with it.

from pathlib import Path
from huggingface_hub import HfApi

REPO_ID = "SandyVeliz/acervo-extractor-v3"
GGUF_DIR = Path("./outputs/s1_sft_v3/gguf")
LORA_DIR = Path("./outputs/s1_sft_v3/final_lora")

api = HfApi()
info = api.whoami()
print(f"Authenticated as: {info['name']}")

# Create repo
api.create_repo(REPO_ID, exist_ok=True, repo_type="model")
print(f"Repo: https://huggingface.co/{REPO_ID}")

# Upload LoRA adapter
print("
Uploading LoRA adapter...")
for f in LORA_DIR.glob("*"):
    if f.is_file() and f.suffix in (".json", ".safetensors", ".jinja"):
        api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name, repo_id=REPO_ID)
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

# Upload GGUF (only quantized, skip 18GB bf16)
print("
Uploading GGUF files...")
for f in GGUF_DIR.glob("*.gguf"):
    if "bf16" not in f.name:
        api.upload_file(path_or_fileobj=str(f), path_in_repo=f"gguf/{f.name}", repo_id=REPO_ID)
        print(f"  gguf/{f.name} ({f.stat().st_size / 1e9:.1f} GB)")

print(f"
Done! https://huggingface.co/{REPO_ID}")